# 02 — Flash Responses (FAFB T4 Pathway)

Simulate full-field ON and OFF flash stimuli delivered to photoreceptors
and record downstream responses across the T4 pathway using **real FAFB
connectome** data.

1. Build network (FAFB pathway)
2. Generate ON / OFF flash stimuli
3. Visualise stimuli
4. Simulate & plot individual neuron responses
5. Flash-response index (FRI) analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from neuro_framework.models import (
    load_or_build_cached_net,
    apply_postbuild_parameter_overrides,
    build_pathway_override_rules,
)

os.makedirs('../logs', exist_ok=True)
print(f'PyTorch {torch.__version__}')


## 1. Load Current Optic-Lobe Net

This notebook now reuses the current optic-lobe cached `FAFBMCNetwork`, then applies the same post-build override rules as the main optic-lobe notebook before running flash stimuli.


In [ ]:
DATA_DIR = '../../mcHH/data/optic_lobe_right'
MORPH_PKG_DIR = '../../mcHH/data/optic_lobe_type_packages_v1'
ION_RULES = '../data/ion_channel_rules.csv'
SYN_RULES = '../data/synapse_rules.csv'
NT_ION_RULES = None
ROOT_ION_OVERRIDES = None

DT = 0.1
NCOMP = 2
MIN_SYN_COUNT = 3
MORPH_PROGRESS_EVERY = 5000

R16_ELEAK_SHIFT_MV = -10.0
R16_VTH_SHIFT_MV = 8.0
R16_TARGET_VTH_SHIFTS_MV = {'L1': -4.0, 'L3': -4.0}
R16_TARGET_GS_GAINS = {'L1': 1.5, 'L3': 1.25}

CACHE_DIR = '../../mcHH/data/cache'
NET_CACHE_PATH = f'{CACHE_DIR}/optic_lobe_net_type_rules_v3.pt'
FORCE_REBUILD = False
SAVE_CACHE_AFTER_BUILD = True

cache_meta = {
    'data_dir': DATA_DIR,
    'morph_pkg_dir': MORPH_PKG_DIR,
    'ion_rules': ION_RULES,
    'syn_rules': SYN_RULES,
    'dt': DT,
    'ncomp': NCOMP,
    'min_syn_count': MIN_SYN_COUNT,
}

net, loaded_cache_meta, loaded_from_cache, build_elapsed = load_or_build_cached_net(
    cache_path=NET_CACHE_PATH,
    force_rebuild=FORCE_REBUILD,
    save_cache_after_build=SAVE_CACHE_AFTER_BUILD,
    cache_meta=cache_meta,
    data_dir=DATA_DIR,
    morphology_package_dir=MORPH_PKG_DIR,
    ion_rules_path=ION_RULES,
    syn_rules_path=SYN_RULES,
    nt_ion_rules_path=NT_ION_RULES,
    neuron_ion_overrides_path=ROOT_ION_OVERRIDES,
    dt=DT,
    ncomp=NCOMP,
    min_syn_count=MIN_SYN_COUNT,
    morphology_progress_every=MORPH_PROGRESS_EVERY,
)
neuron_override_rules, synapse_override_rules = build_pathway_override_rules(
    eLeak_shift_mV=R16_ELEAK_SHIFT_MV,
    v_th_shift_mV=R16_VTH_SHIFT_MV,
    target_v_th_shifts_mV=R16_TARGET_VTH_SHIFTS_MV,
    target_gs_gains=R16_TARGET_GS_GAINS,
)
net = apply_postbuild_parameter_overrides(
    net, neuron_rules=neuron_override_rules, synapse_rules=synapse_override_rules, reset_first=True
)
print('loaded_from_cache =', loaded_from_cache)
print(f'build_elapsed = {build_elapsed:.1f}s')
print(net)
n_photo = int(net.input_mask.sum().item())
print(f'Photoreceptors: {n_photo}')


## 2. Generate Flash Stimuli

- **ON flash**: baseline → bright → baseline  
- **OFF flash**: bright baseline → dark → bright baseline

Ion channel conductances are ~1-3 mS/cm², so we use I=30 μA/cm²
for a ~15 mV photoreceptor response.

In [ ]:
dt = DT
t_pre = 100.0
t_stim = 200.0
t_post = 200.0
T_ms = t_pre + t_stim + t_post
T_steps = int(T_ms / dt)
t = np.arange(T_steps) * dt

stim_on = int(t_pre / dt)
stim_off = int((t_pre + t_stim) / dt)
I_amp = 30.0  # μA/cm²

x_on = torch.zeros(1, T_steps, n_photo)
x_on[0, stim_on:stim_off, :] = I_amp

x_off = torch.full((1, T_steps, n_photo), I_amp)
x_off[0, stim_on:stim_off, :] = 0.0

print(f'Stimulus: {T_ms:.0f} ms, flash={t_pre:.0f}-{t_pre+t_stim:.0f} ms, I={I_amp} μA/cm², dt={dt}')


## 3. Visualise Stimuli

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
for ax, (label, x) in zip(axes, [('ON Flash', x_on), ('OFF Flash', x_off)]):
    ax.fill_between(t, 0, x[0, :, 0].numpy(), alpha=0.3, color='orange')
    ax.plot(t, x[0, :, 0].numpy(), color='orange', linewidth=1.5)
    ax.set_title(label)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('I_ext (μA/cm²)')
    ax.axvline(t_pre, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(t_pre + t_stim, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../logs/flash_stimuli.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Simulate & Plot Responses

Each row shows individual neuron traces (thin lines) and population mean (thick).
Only a subset of R1-R6 connect to each downstream type, so responses vary.

In [ ]:
with torch.no_grad():
    V_on = net(x_on, dt=dt)
    V_off = net(x_off, dt=dt)

print(f'V_on range: [{V_on.min():.1f}, {V_on.max():.1f}] mV')

In [ ]:
pathway = ['R1-6', 'R8', 'L1', 'L3', 'L5', 'Mi1', 'Mi9', 'Mi4', 'Tm3', 'T4a']

fig = plt.figure(figsize=(14, 2.5 * len(pathway)))
gs = GridSpec(len(pathway), 2, figure=fig, hspace=0.4, wspace=0.15)

for row, ct in enumerate(pathway):
    idx = net.get_indices_by_type(ct)
    if not idx:
        continue
    for col, (V, label) in enumerate([(V_on, 'ON'), (V_off, 'OFF')]):
        ax = fig.add_subplot(gs[row, col])
        vs = V[0, :, idx].numpy()
        # Individual traces
        n_show = min(10, len(idx))
        for j in range(n_show):
            ax.plot(t, vs[:, j], alpha=0.3, linewidth=0.5, color=f'C{row}')
        # Mean trace
        ax.plot(t, vs.mean(axis=1), color=f'C{row}', linewidth=2.0)
        ax.axvline(t_pre, color='gray', linestyle='--', alpha=0.3)
        ax.axvline(t_pre + t_stim, color='gray', linestyle='--', alpha=0.3)
        ax.set_ylabel(f'{ct} (n={len(idx)})', fontsize=9)
        if row == 0:
            ax.set_title(f'{label} Flash', fontsize=11)
        if row == len(pathway) - 1:
            ax.set_xlabel('Time (ms)')

plt.savefig('../logs/flash_responses.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Flash Response Index

In [ ]:
baseline_sl = slice(0, stim_on)
stim_sl = slice(stim_on + int(50/dt), stim_off)  # skip transient

def compute_response(V, bl, sl):
    return V[0, sl].mean(dim=0) - V[0, bl].mean(dim=0)

R_on = compute_response(V_on, baseline_sl, stim_sl)
R_off = compute_response(V_off, baseline_sl, stim_sl)
eps = 0.1
FRI = (R_on - R_off) / (R_on.abs() + R_off.abs() + eps)

# Print per-type stats
types_to_show = ['R1-6', 'R7', 'R8', 'L1', 'L2', 'L3', 'L4', 'L5',
                 'Mi1', 'Mi4', 'Mi9', 'Tm3', 'T4a', 'T4b']
print(f'{"Type":>10s}  {"ON resp":>8s} {"OFF resp":>8s} {"FRI":>6s}')
for ct in types_to_show:
    idx = net.get_indices_by_type(ct)
    if idx:
        on_r = R_on[idx].mean().item()
        off_r = R_off[idx].mean().item()
        fri = FRI[idx].mean().item()
        print(f'{ct:>10s}  {on_r:8.2f} {off_r:8.2f} {fri:6.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
positions = []
labels = []
plot_types = ['R1-6', 'R8', 'L1', 'L3', 'L5', 'Mi1', 'Mi9', 'Mi4', 'Tm3', 'T4a', 'T4b']
for i, ct in enumerate(plot_types):
    idx = net.get_indices_by_type(ct)
    if not idx:
        continue
    fri_vals = FRI[idx].numpy()
    bp = ax.boxplot(fri_vals, positions=[i], widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor=f'C{i % 10}', alpha=0.5))
    labels.append(f'{ct}\n(n={len(idx)})')
    positions.append(i)

ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_ylabel('FRI')
ax.set_title('Flash Response Index (ON vs OFF) — FAFB T4 Pathway')
ax.set_ylim(-1.1, 1.1)
plt.tight_layout()
plt.savefig('../logs/flash_fri.png', dpi=150, bbox_inches='tight')
plt.show()